# 07 · Case study: Waterbirds — does our fix work on a standard shortcut benchmark?
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mundmina/dl_project/blob/main/notebooks/07_case_waterbirds.ipynb)

**Runtime → T4 GPU → Run all.** ~1.5–2 ч: скачивание (~480 MB, один раз), маски (~10 мин, один раз), 2 модели × 3 seed.

Background bias на породах — маленький удобный случай большой проблемы shortcut learning. Здесь проверяем,
переносится ли наш метод на **Waterbirds** (Sagawa et al., 2020): птицы из CUB на фонах Places, и в train
95% водоплавающих птиц сидят на воде, а 95% сухопутных — на суше. Модель легко учится отличать воду от суши
вместо птиц. Главная метрика — **worst-group accuracy**: точность на самой трудной группе (обычно «водоплавающая
птица на суше»).

- `waterbirds_A` — обычное дообучение ResNet-18;
- `waterbirds_B` — наш метод без изменений: с вероятностью 0.5 фон заменяется случайной сценой Places365.
  Ручных масок нет — их делает сеть сегментации DeepLabV3 (класс «bird»).

Checkpoint выбирается по средней точности на val, без групповых меток (честное сравнение).

In [ ]:
# Shared setup: Drive, repo, dependencies. Same cell in every notebook.
# Colab: nothing to set. Local machine (own GPU): see LOCAL.md — open this notebook from the repo's notebooks/
# folder and set BGBIAS_DRIVE, BGBIAS_DATA and GITHUB_TOKEN before starting Jupyter.
REPO_URL = "https://github.com/mundmina/dl_project.git"
BRANCH = "main"

import os, subprocess, sys
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_DIR = "/content/dl_project"
    drive.mount("/content/drive")
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
    else:  # always run the latest code from GitHub
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"], check=True)
else:
    REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))   # notebook opened from <repo>/notebooks
    assert os.path.exists(os.path.join(REPO_DIR, "src", "bgbias")), "open the notebook from the repo's notebooks/ folder"
    missing = [v for v in ("BGBIAS_DRIVE", "BGBIAS_DATA") if not os.environ.get(v)]
    assert not missing, f"set {missing} before starting Jupyter (see LOCAL.md)"
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], check=True)
commit = subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout
print("code version:", commit)
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")
# DataLoader workers on Windows start fresh interpreters: they find bgbias through PYTHONPATH, not sys.path
os.environ["PYTHONPATH"] = os.pathsep.join(p for p in (f"{REPO_DIR}/src", os.environ.get("PYTHONPATH")) if p)
try:  # ask for secret access now, not at the end of a long run (the prompt times out unattended)
    if not os.environ.get("GITHUB_TOKEN"):
        from google.colab import userdata
        userdata.get("GITHUB_TOKEN")
    print("GITHUB_TOKEN: ok")
except Exception as e:  # noqa: BLE001
    print(f"GITHUB_TOKEN not available ({type(e).__name__}): results will not be pushed to GitHub")

In [ ]:
import torch
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU"

In [ ]:
from pathlib import Path
from bgbias.project import load_project_data
from bgbias.utils import DRIVE_ROOT

data = load_project_data(REPO_DIR)   # splits / manifests from the repo or from MyDrive/bgbias/week1
RUNS = DRIVE_ROOT / "runs"           # checkpoints and per-run outputs live on Drive
RUNS.mkdir(parents=True, exist_ok=True)
{name: len(df) for name, df in data.splits.items()}

In [ ]:
import numpy as np
import pandas as pd
from bgbias import waterbirds as W
from bgbias.train import load_config

device = torch.device("cuda")
WB_ROOT = W.prepare_waterbirds()
wb = W.load_metadata(WB_ROOT)
print(pd.crosstab([wb.split, wb.y.map({0: "landbird", 1: "waterbird"})], wb.place.map({0: "land", 1: "water"})))
masks = W.compute_masks(WB_ROOT, wb, device, DRIVE_ROOT / "data" / "waterbirds_masks.npz")
found = np.mean([W.unpack_mask(m).mean() > 0.005 for m in masks.values()])
print(f"segmentation found the bird in {found:.1%} of images")

In [ ]:
import matplotlib.pyplot as plt
from bgbias.datasets import build_transform

bg_pool = data.bg_pool("train")
demo = W.WaterbirdsDataset(wb[wb.split == "train"].sample(6, random_state=0), WB_ROOT, lambda im: im, masks,
                           p_swap=1.0, bg_pool=bg_pool, places_root=data.places_root)
torch.manual_seed(0)
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for i in range(6):
    axes[0, i].imshow(W.compose.load_image(WB_ROOT / demo.df.at[i, "img_filename"]))
    axes[0, i].set_title(W.GROUPS[demo.df.at[i, "group"]], fontsize=9)
    axes[1, i].imshow(demo.render(i))
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
CONFIGS = ["waterbirds_A", "waterbirds_B"]
SEEDS = [0, 1, 2]
WB_RUNS = DRIVE_ROOT / "runs_waterbirds"   # kept apart: 04 / 04b / 05b scan runs/ for 37-class Pet models
configs = {name: load_config(f"{REPO_DIR}/configs/{name}.yaml") for name in CONFIGS}
for seed in SEEDS:
    for name, cfg in configs.items():
        run_dir = WB_RUNS / f"{name}_seed{seed}"
        print(f"=== {run_dir.name} ===")
        W.train(cfg, seed, wb, WB_ROOT, masks, data.places_root, bg_pool, run_dir, device)

## Test: средняя точность и точность по группам

In [ ]:
from bgbias.models import build_model

test_loader = W._loader(W.WaterbirdsDataset(wb[wb.split == "test"], WB_ROOT, build_transform(train=False)), 128, False, 0)
rows = []
for name in CONFIGS:
    for seed in SEEDS:
        ckpt = torch.load(WB_RUNS / f"{name}_seed{seed}" / "best.pt", map_location=device, weights_only=False)
        model = build_model(ckpt["config"].get("arch", "resnet18"), 2, pretrained=False).to(device)
        model.load_state_dict(ckpt["model"])
        rows.append({"model": name, "seed": seed, **W.group_metrics(*W.predict(model, test_loader, device))})
res = pd.DataFrame(rows)
OUT = Path(REPO_DIR) / "results" / "waterbirds"
OUT.mkdir(parents=True, exist_ok=True)
res.to_csv(OUT / "test_per_run.csv", index=False)
summary = res.drop(columns="seed").groupby("model").agg(["mean", "std"])
summary.to_csv(OUT / "test_summary.csv")
summary.round(3)

In [ ]:
from matplotlib.ticker import PercentFormatter
from bgbias.plotting import COLORS, saver, use_style

use_style()
FIG_DIR = Path(REPO_DIR) / "report" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
save = saver(FIG_DIR)

cols = ["avg_acc", "worst_group_acc", *W.GROUPS.values()]
names = ["Average", "Worst group", *W.GROUPS.values()]
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(cols))
for name, key, dx in (("waterbirds_A", "resnet_A", -0.18), ("waterbirds_B", "resnet_B", 0.18)):
    m = res[res.model == name][cols]
    ax.bar(x + dx, m.mean(), width=0.34, yerr=m.std(), capsize=3, color=COLORS[key],
           label="A: plain fine-tuning" if name.endswith("A") else "B: random background swap (ours)")
    for xi, v in zip(x + dx, m.mean()):
        ax.annotate(f"{v:.0%}", (xi, v), xytext=(0, 4), textcoords="offset points", ha="center", fontsize=8)
ax.set_xticks(x, names, fontsize=9)
ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
ax.set_ylim(0, 1.05)
ax.set_ylabel("Test accuracy (mean ± std, 3 seeds)")
ax.set_title("Waterbirds: background swap and the hardest groups")
ax.legend(loc="lower left")
save(fig, "fig9_waterbirds")

In [ ]:
import shutil
from bgbias.github_sync import sync

for name in CONFIGS:
    for seed in SEEDS:
        src = WB_RUNS / f"{name}_seed{seed}" / "history.json"
        dst = OUT / "runs" / f"{name}_seed{seed}.json"
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(src, dst)
shutil.copytree(OUT, DRIVE_ROOT / "results" / "waterbirds", dirs_exist_ok=True)
shutil.copytree(FIG_DIR, DRIVE_ROOT / "figures", dirs_exist_ok=True)
sync(REPO_DIR, "Case study: Waterbirds")